## 工廠監控 : Python 與 Modbus通訊

----


### 為何在工控應用中導入Python

- 因為工作上需做的產品會從linux kernel, driver, fs porting，一直到firmware撰寫，script，網頁設定頁面的開發。

- C + bash   + C      + PHP    + html + javascript

- C + python + python + python + html + javascript + CSS

- 有時進一步還要開發Server端的程式，處理資料庫…等事項


### 今天的主軸：如何使用python套件和PLC、電表溝通的經驗應用分享
    
- ![PLC_Meter](image/plc_meter.png)

### 工廠監控流程

![工廠監控流程](image/監控流程.png)


- 「自動化」：感測器 + 控制器

- 「監控」：感測器 + 控制器 + Server

- 「生產履歷」：感測器 + 控制器 + Server + 資料視覺化

- 「智慧生產」：感測器 + 控制器 + Server + (資料視覺化) + 資料分析 + 自我調整


### Python在工業4.0中的角色

- 最常聽到的應用：「Big Data」、「資料視覺化」、「數據分析」

- ![監控流程2](image/監控流程2.png)

- 控制器? 資料收集器? , Python當然也可以勝任!


----

## 工廠監控的現況

看看大家現在怎麼做…

----


### 很常見的方式

- SCADA + HMI + PLC + IO module

- PLC + SCADA

### 系統小一點的話

- HMI + PLC + IO module


### 喜歡自己DIY?

- 自己開發! VB.Net, C#, C 各種語言不拘!

### 為何SCADA這麼常出現?

- 因為工業控制需要快速又穩定的RAD工具，讓每個人都能做出一定水準之上的系統

![Scada_std_anim_no_lang](image/Scada_std_anim_no_lang.gif)


----

## Modbus

一個工業控制一定不能錯過的協定

----


### 何謂 Modbus?

一種工業控制中很常用的通訊協定

- [維基百科怎麼說](https://zh.wikipedia.org/wiki/Modbus)?

- 為何介紹Modbus
       

### modbus格式介紹 

[參考](http://gridconnect.com/blog/tag/modbus-rtu/)

- Modbus/RTU: [start time] [Address 8bits + Function 8bits + Data Nx8bits + CRC 16bits] [End time]

- Modbus/TCP: [header 6byte + Address 8bits + Function 8bits + Data Nx8bits]

- Modbus/ASCII: 現在比較少人用，跳過不講

![格式](image/MODBUS-Frame.png)
        


### modbus常被採用的原因
        
- 公開發表並且無版稅要求

- 相對容易的工業網路部署

- 協定格式簡單，極省資源


----

## Modbus套件

----



### 網路上較多人提到的三個套件

[performance比較](https://stackoverflow.com/questions/17081442/python-modbus-library)

- modbus-tk: Modbus/RTU, Modbus/TCP

- pymodbus: 據說實作最完整，但使用資源相對的多，相依套件也多

- MinimalModbus: Modbus/RTU, Modbus/ASCII



### Modbus-tk

個人推薦使用

- 安裝方式
    - pip install serial
    - pip install modbus_tk

- 操作
    - [Python與PLC共舞](https://github.com/maloyang/PLC-Python)
     

# 應該快睡著了…

## 先來秀一下程式動起來的效果吧


----

## PLC

工業控制常用的元素，[看看wiki怎麼說](https://zh.wikipedia.org/wiki/%E5%8F%AF%E7%BC%96%E7%A8%8B%E9%80%BB%E8%BE%91%E6%8E%A7%E5%88%B6%E5%99%A8)

----



### 通訊方式

- 自有協定

- <h3>Modbus</h3>

- CAN

- ...etc

![protocol_cloud](image/protocol_cloud.png)

### PLC的Modbus點表

![PLC點表](image/fatek_modbus_addr.png)

- [Live Demo](Modbus.ipynb)

### Modbus/TCP Demo

In [1]:
import serial
import modbus_tk
import modbus_tk.defines as cst
import modbus_tk.modbus_rtu as modbus_rtu
import modbus_tk.modbus_tcp as modbus_tcp
import time


In [2]:

mb_ip = '192.168.8.139'
mb_port = 502
timeout = 1000
master = modbus_tcp.TcpMaster(mb_ip, mb_port)
master.set_timeout(timeout/1000.0)

mbId = 1
addr = 0 #base0 --> my 110V Led燈泡

for i in range(1, 7):
    try:
        value = i%2
        #-- FC5: write multi-coils
        rr = master.execute(mbId, cst.WRITE_SINGLE_COIL, addr, output_value=value)
        print("Write(addr, value)=",  rr)

    except Exception as e:
        print("modbus test Error: " + str(e))

    time.sleep(2)

master._do_close()


Write(addr, value)= (0, 65280)
Write(addr, value)= (0, 0)
Write(addr, value)= (0, 65280)
Write(addr, value)= (0, 0)
Write(addr, value)= (0, 65280)
Write(addr, value)= (0, 0)


True

### Modbus/RTU Demo

In [2]:
import serial
import modbus_tk
import modbus_tk.defines as cst
import modbus_tk.modbus_rtu as modbus_rtu
import time


In [3]:
mbComPort = 'COM5'   # your RS-485 port. for linux --> "/dev/ttyUSB0"
baudrate = 9600
databit = 8 #7, 8
parity = 'N' #N, O, E
stopbit = 1 #1, 2
mbTimeout = 100 # ms


In [4]:

mb_port = serial.Serial(port=mbComPort, baudrate=baudrate, bytesize=databit, parity=parity, stopbits=stopbit)
master = modbus_rtu.RtuMaster(mb_port)
master.set_timeout(mbTimeout/1000.0)

mbId = 1
addr = 0 #base0 --> my 110V Led燈泡

for i in range(5):
    try:
        value = i%2
        #-- FC5: write multi-coils
        rr = master.execute(mbId, cst.WRITE_SINGLE_COIL, addr, output_value=value)
        print("Write(addr, value)=",  rr)

    except Exception as e:
        print("modbus test Error: " + str(e))

    time.sleep(2)

master._do_close()


Write(addr, value)= (2, 0)
Write(addr, value)= (2, 65280)
Write(addr, value)= (2, 0)
Write(addr, value)= (2, 65280)
Write(addr, value)= (2, 0)


True

----

## Power Meter

電力監控常見的元素

----



### Power Meter的點表

![Power Meter的點表](image/power_meter.gif)



### Power Meter的浮點數表示方式

![浮點數](image/power_meter_float.gif)

----
![float](image/mb_float.png)
- [Live Demo](Modbus.ipynb)


### Modbus/TCP Demo

## Modbus/TCP: 電表


In [6]:
import serial
import modbus_tk
import modbus_tk.defines as cst
import modbus_tk.modbus_rtu as modbus_rtu
import modbus_tk.modbus_tcp as modbus_tcp
from struct import *
import time


In [7]:

mb_ip = '192.168.8.139'
mb_port = 503
timeout = 1000
master = modbus_tcp.TcpMaster(mb_ip, mb_port)
master.set_timeout(timeout/1000.0)

mbId = 2
addr = 0x1000 # power-meter is base 0, 0x1000=4096
# notice: meter not support FC6, only FC16

try:
    # FC3
    rr = master.execute(mbId, cst.READ_INPUT_REGISTERS, addr, 2)
    print('read value:', rr)

    # convert to float:
    # IEEE754 ==> (Hi word Hi Bite, Hi word Lo Byte, Lo word Hi Byte, Lo word Lo Byte)
    try:
        v_a_hi = rr[1]
        v_a_lo = rr[0]
        v_a = unpack('>f', pack('>HH', v_a_hi, v_a_lo))
        print('v_a=', v_a)
        #struct.unpack(">f",'\x42\xd8\x6b\x8d')
    except Exception as e:
        print(e)

except Exception as e:
    print("modbus test Error: " + str(e))


master._do_close()



read value: (40448, 17124)
v_a= (114.30859375,)


True

### Modbus/RTU

In [6]:
import serial
import modbus_tk
import modbus_tk.defines as cst
import modbus_tk.modbus_rtu as modbus_rtu
import time
from struct import *

In [7]:
mbComPort = 'COM5' #your RS-485 port #'/dev/ttyUSB0' for linux(RPi3)
baudrate = 9600
databit = 8
parity = 'N'
stopbit = 1
mbTimeout = 100 # ms

In [35]:
0x1002

4098

In [18]:
mb_port = serial.Serial(port=mbComPort, baudrate=baudrate, bytesize=databit, parity=parity, stopbits=stopbit)
master = modbus_rtu.RtuMaster(mb_port)
master.set_timeout(mbTimeout/1000.0)

mbId = 4
addr = 0x1000 # power-meter is base 0
# notice: meter not support FC6, only FC16

try:
    # FC3
    rr = master.execute(mbId, cst.READ_INPUT_REGISTERS, addr, 2)
    print('read value:', rr)

    # convert to float:
    # IEEE754 ==> (Hi word Hi Bite, Hi word Lo Byte, Lo word Hi Byte, Lo word Lo Byte)
    try:
        v_a_hi = rr[1]
        v_a_lo = rr[0]
        v_a = unpack('>f', pack('>HH', v_a_hi, v_a_lo))
        print('v_a=', v_a)
        #struct.unpack(">f",'\x42\xd8\x6b\x8d')
    except Exception as e:
        print(e)

except Exception as e:
    print("modbus test Error: " + str(e))


master._do_close()


read value: (39322, 17120)
v_a= (112.30000305175781,)


True

In [22]:
0x1020

4128

In [25]:
mb_port = serial.Serial(port=mbComPort, baudrate=baudrate, bytesize=databit, parity=parity, stopbits=stopbit)
master = modbus_rtu.RtuMaster(mb_port)
master.set_timeout(mbTimeout/1000.0)

mbId = 4
addr = 0x1034 #kWh

try:
    # FC3
    rr = master.execute(mbId, cst.READ_INPUT_REGISTERS, addr, 2)
    print('read value:', rr)

    # convert to float:
    # IEEE754 ==> (Hi word Hi Bite, Hi word Lo Byte, Lo word Hi Byte, Lo word Lo Byte)
    try:
        hi = rr[1]
        lo = rr[0]
        kwh = unpack('>f', pack('>HH', hi, lo))
        print('kWh=', kwh)
    except Exception as e:
        print(e)

except Exception as e:
    print("modbus test Error: " + str(e))


master._do_close()


read value: (41779, 17663)
kWh= (2045.0999755859375,)


True

In [27]:
mb_port = serial.Serial(port=mbComPort, baudrate=baudrate, bytesize=databit, parity=parity, stopbits=stopbit)
master = modbus_rtu.RtuMaster(mb_port)
master.set_timeout(mbTimeout/1000.0)

mbId = 4
#[0x1000-0x1001]=VIn_a
addr = 0x1000#4096

try:
    # FC3
    rr = master.execute(mbId, cst.READ_INPUT_REGISTERS, addr, 26)
    print('read value:', rr)

    # convert to float:
    # IEEE754 ==> (Hi word Hi Bite, Hi word Lo Byte, Lo word Hi Byte, Lo word Lo Byte)
    try:
        # VIn_a
        v_a_hi = rr[1]
        v_a_lo = rr[0]
        v_a = unpack('>f', pack('>HH', v_a_hi, v_a_lo))
        print('v_a=', v_a)
        #struct.unpack(">f",'\x42\xd8\x6b\x8d')
        
        # VIn_avg
        v_hi = rr[7]
        v_lo = rr[6]
        v_avg = unpack('>f', pack('>HH', v_hi, v_lo))
        print('v_avg=', v_avg)
        
        # Frequency
        freq_hi = rr[25]
        freq_lo = rr[24]
        freq = unpack('>f', pack('>HH', freq_hi, freq_lo))
        print('Frequency=', freq)
        
    except Exception as e:
        print(e)

except Exception as e:
    print("modbus test Error: " + str(e))


master._do_close()


read value: (39658, 17131, 0, 0, 0, 0, 39658, 17131, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 19100, 17008)
v_a= (117.80256652832031,)
v_avg= (117.80256652832031,)
Frequency= (60.07286071777344,)


True

----

## MQTT - 即時性應用

已經可以採集資料了，來談談怎麼上傳雲端吧

----


### Mosquitto

一個broker套件，當然也可以做為client使用

- 以NB X260來說，4000多個連結沒有問題

![Mosquitto](image/Eclipse-Mosquitto-logo.png)


### paho

便利的MQTT client端套件

- [link](https://pypi.org/project/paho-mqtt/)
- install: `pip install paho-mqtt`

![paho](image/mqtt-paho-featured-image.jpg)


### live demo

- 以先架好的rpi3控制PLC為例


In [29]:
import paho.mqtt.client as mqtt  #import the client1
import time

import serial
import modbus_tk
import modbus_tk.defines as cst
import modbus_tk.modbus_rtu as modbus_rtu
import time

mbComPort = 'COM5'
baudrate = 9600
databit = 8
parity = 'N'
stopbit = 1
mbTimeout = 100 # ms

def on_connect(client, userdata, flags, rc):
    m="Connected flags"+str(flags)+", result code "+str(rc)+", client_id  "+str(client)
    print(m)

    # first value OFF
    print('set light off!')
    control_light(0)
    client1.publish(topic,0)    

def on_message(client1, userdata, message):
    print("message received  "  ,str(message.payload.decode("utf-8")), message.topic, message.qos, message.retain)
    if message.topic == topic:
        my_message = str(message.payload.decode("utf-8"))
        print("set light: ", my_message)
        if my_message=='1' or my_message==1:
            control_light(1)
        else:
            control_light(0)

def on_log(client, userdata, level, buf):
    print("log: ",buf)

def control_light(value):
    mb_port = serial.Serial(port=mbComPort, baudrate=baudrate, bytesize=databit, parity=parity, stopbits=stopbit)
    master = modbus_rtu.RtuMaster(mb_port)
    master.set_timeout(mbTimeout/1000.0)

    mbId = 1
    addr = 2 #base0

    try:
        #-- FC5: write multi-coils
        rr = master.execute(mbId, cst.WRITE_SINGLE_COIL, addr, output_value=value)
        print("Write(addr, value)=%s" %(str(rr)))

    except Exception as e:#except Exception, e:
        print("modbus test Error: " + str(e))

    master._do_close()


# some online free broker:
#   iot.eclipse.org
#   test.mosquitto.org
#   broker.hivemq.com
broker_address="iot.eclipse.org"
topic = "malo-iot/light"
client1 = mqtt.Client()    #create new instance
client1.on_connect = on_connect        #attach function to callback
client1.on_message = on_message        #attach function to callback
#client1.on_log=on_log

time.sleep(1)
client1.connect("iot.eclipse.org", 1883, 60)      #connect to broker
#client1.loop_start()    #start the loop
client1.subscribe(topic)

client1.loop_forever()

Connected flags{'session present': 0}, result code 0, client_id  <paho.mqtt.client.Client object at 0x0000000004ED3A90>
set light off!
Write(addr, value)=(2, 0)
message received   1 malo-iot/light 0 1
set light:  1
Write(addr, value)=(2, 65280)
message received   0 malo-iot/light 0 0
set light:  0
Write(addr, value)=(2, 0)
message received   1 malo-iot/light 0 0
set light:  1
Write(addr, value)=(2, 65280)
message received   0 malo-iot/light 0 0
set light:  0
Write(addr, value)=(2, 0)
message received   1 malo-iot/light 0 0
set light:  1
Write(addr, value)=(2, 65280)
message received   0 malo-iot/light 0 0
set light:  0
Write(addr, value)=(2, 0)
message received   1 malo-iot/light 0 0
set light:  1
Write(addr, value)=(2, 65280)


KeyboardInterrupt: 

## MQTT demo

In [9]:
!pip install paho-mqtt

  Using cached paho_mqtt-2.1.0-py3-none-any.whl.metadata (23 kB)
Using cached paho_mqtt-2.1.0-py3-none-any.whl (67 kB)



[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
import paho.mqtt.client as mqtt  #import the client1
import time

import time

def on_connect(client, userdata, flags, rc):
    m="Connected flags"+str(flags)+", result code "+str(rc)+", client_id  "+str(client)
    print(m)


def on_message(client1, userdata, message):
    print("message received  "  ,str(message.payload.decode("utf-8")), message.topic, message.qos, message.retain)
    if message.topic == topic:
        my_message = str(message.payload.decode("utf-8"))

def on_log(client, userdata, level, buf):
    print("log: ",buf)


# some online free broker:
#   iot.eclipse.org
#   test.mosquitto.org
#   broker.hivemq.com
broker_address="broker.hivemq.com"
topic = "malo-iot/msg"
client1 = mqtt.Client()    #create new instance
client1.on_connect = on_connect        #attach function to callback
client1.on_message = on_message        #attach function to callback
#client1.on_log=on_log

time.sleep(1)
client1.connect(broker_address, 1883, 60)      #connect to broker
#client1.loop_start()    #start the loop
client1.subscribe(topic)

for k in range(6):
    client1.publish(topic, k)
    time.sleep(3)

time.sleep(3)
#client1.loop_forever()

C:\Users\Malo\AppData\Local\Temp\ipykernel_28196\1621275835.py:26: DeprecationWarning: Callback API version 1 is deprecated, update to latest version
  client1 = mqtt.Client()    #create new instance


In [12]:
# new demo , 2026-06-16

import time
import paho.mqtt.client as mqtt

# 1. 定義 Callback 函式
# 注意：paho-mqtt 2.0 的 on_connect 參數順序與欄位有所轉變，這裡加上了 reason_code 與 properties
def on_connect(client, userdata, flags, reason_code, properties):
    print(f"連線狀態: {reason_code}")
    if reason_code == 0:
        print("連線成功！開始訂閱主題...")
        # 建議在連線成功後再進行訂閱
        client.subscribe(topic)
    else:
        print(f"連線失敗，錯誤碼: {reason_code}")

def on_message(client, userdata, message):
    payload = message.payload.decode("utf-8")
    print(f"收到訊息 -> 主題: {message.topic}, 內容: {payload}, QoS: {message.qos}")

def on_log(client, userdata, level, buf):
    print("Log: ", buf)

# 2. 設定參數
broker_address = "broker.hivemq.com"
topic = "malo-iot/msg"

# 3. 建立實例 (關鍵修正：必須指定 CallbackAPI version)
# 使用 CallbackAPIVersion.VERSION2 是新版標準
client1 = mqtt.Client(callback_api_version=mqtt.CallbackAPIVersion.VERSION2)

# 綁定 Callback 函式
client1.on_connect = on_connect
client1.on_message = on_message
# client1.on_log = on_log

# 4. 連線與啟動網路循環處理 Loop
print("正在連線到 Broker...")
client1.connect(broker_address, 1883, 60)

# 關鍵修正：必須啟動 loop，Paho 才會在背景處理連線、發送與接收訊息
client1.loop_start()

# 稍等一下確保連線完成
time.sleep(1)

# 5. 模擬電表發送資料
for k in range(6):
    meter_data = f"Meter Value: {k}"
    print(f"正在發送: {meter_data}")
    
    # publish 的內容建議轉成字串或 bytes
    client1.publish(topic, payload=meter_data, qos=1) 
    time.sleep(2)

# 留點時間讓最後一筆訊息收發完畢
time.sleep(2)

# 6. 結束程式前關閉 loop
client1.loop_stop()
client1.disconnect()
print("程式結束")

正在連線到 Broker...
正在發送: Meter Value: 0
連線狀態: Success
連線成功！開始訂閱主題...
正在發送: Meter Value: 1
收到訊息 -> 主題: malo-iot/msg, 內容: Meter Value: 1, QoS: 0
正在發送: Meter Value: 2
收到訊息 -> 主題: malo-iot/msg, 內容: Meter Value: 2, QoS: 0
正在發送: Meter Value: 3
收到訊息 -> 主題: malo-iot/msg, 內容: Meter Value: 3, QoS: 0
正在發送: Meter Value: 4
收到訊息 -> 主題: malo-iot/msg, 內容: Meter Value: 4, QoS: 0
正在發送: Meter Value: 5
收到訊息 -> 主題: malo-iot/msg, 內容: Meter Value: 5, QoS: 0
程式結束


### MQTT測試的網頁工具
- https://www.hivemq.com/demos/websocket-client/
- 

### 自己做restful API + PostgreSQL


### ThingSpeak

- 上傳資料自動畫成趨勢圖


----

### 運用 ThingSpeak收集資料

demo

----

In [5]:
# push data

import requests

# field1: T
# field2: H
url = 'https://api.thingspeak.com/update'
api_key = 'RD1VBDRERV09LPV4'
field1 = 24.58
field2 = 63.11
data = {'api_key': api_key, 'field1':field1, 'field2':field2}
r = requests.get(url, params=data)
r

<Response [200]>

In [3]:
r.text

'0'

In [7]:
# continue push data

my_h = [65.46, 65.75, 62.49, 65.4, 65.89, 63.11]
my_t = [25.05, 24.72, 25.06, 24.66, 24.99, 24.58]

In [13]:
len(my_t)

6

In [15]:
import requests
import time

for i in range(len(my_t)):
    
    print('---- %s ----' %(i))
    print('H=%s %%, T=%s degree' %(my_h[i], my_t[i]))
    
    # push data
    # field1: T
    # field2: H
    url = 'https://api.thingspeak.com/update'
    api_key = 'RD1VBDRERV09LPV4'
    field1 = my_t[i]
    field2 = my_h[i]
    data = {'api_key': api_key, 'field1':field1, 'field2':field2}
    r = requests.get(url, params=data)
    print(r)
    
    time.sleep(15)
    

---- 0 ----
H=65.46 %, T=25.05 degree
<Response [200]>
---- 1 ----
H=65.75 %, T=24.72 degree
<Response [200]>
---- 2 ----
H=62.49 %, T=25.06 degree
<Response [200]>
---- 3 ----
H=65.4 %, T=24.66 degree
<Response [200]>
---- 4 ----
H=65.89 %, T=24.99 degree
<Response [200]>
---- 5 ----
H=63.11 %, T=24.58 degree
<Response [200]>


----

## Q&A

----